# Evaluate your AI agent with Weave

This notebook shows how to evaluate an AI agent with Weave using the
**Agents / Conversations** workflow — the agent is traced as a conversation of
turns and tool calls (OpenTelemetry spans), and you score it with
`weave.EvaluationLogger`.

You'll:
1. Build and trace a small customer-support agent.
2. Score each run with an LLM judge (task completion as the example score).
3. Compare two agent versions.
4. Score a **multi-turn** conversation.

The agent runs on **Claude Sonnet**; the judge runs on **Claude Opus** — grading
with a stronger, different model than the one under test is good evaluation
practice.

> This notebook makes real LLM calls (agent + judge across several tasks), so it
> will incur provider costs.

## Setup

Install dependencies and provide your API keys.

In [ ]:
%pip install -qU weave anthropic

In [ ]:
import getpass
import os

os.environ["WANDB_API_KEY"] = getpass.getpass("Enter your W&B API key: ")
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

TEAM = input("Enter your W&B team name: ")
PROJECT = input("Enter your W&B project name: ")

In [ ]:
import weave

weave.init(
    f"{TEAM}/{PROJECT}",
    # Hand-instrumenting a bare provider SDK: turn off implicit patching so it
    # doesn't also log each call as a traced Op, duplicating our spans.
    settings={"implicitly_patch_integrations": False},
)

## 1. Build and trace the agent

A minimal customer-support agent with two tools (`lookup_order`, `issue_refund`)
and a policy: refunds are only allowed within 30 days.

The Weave-specific part is the **Conversation SDK** instrumentation inside
`run_agent_turn`: `start_conversation` → `start_turn` → `start_llm` /
`start_tool`, with `llm.record(...)` capturing the model call. It returns the
agent's final reply plus a plain-text transcript of the trajectory for the judge
to read; everything else (the tool backend, the Anthropic tool-use loop, message
conversion) is ordinary agent plumbing.

In [ ]:
import json
import anthropic

# Agent and judge deliberately use different models (good eval practice).
AGENT_MODEL = "claude-sonnet-5"
_client = anthropic.Anthropic()

# --- Mock support backend (tools the agent can call) ---
_ORDERS = {
    "A1001": {"item": "Wireless headphones", "amount": 79.99, "days_since_delivery": 5},
    "A1002": {"item": "Standing desk", "amount": 320.00, "days_since_delivery": 60},
}
REFUND_WINDOW_DAYS = 30

def lookup_order(order_id):
    order = _ORDERS.get(order_id)
    return {"error": f"order {order_id} not found"} if order is None else {"order_id": order_id, **order}

def issue_refund(order_id, amount):
    order = _ORDERS.get(order_id)
    if order is None:
        return {"refunded": False, "reason": "order not found"}
    if order["days_since_delivery"] > REFUND_WINDOW_DAYS:
        return {"refunded": False, "reason": "outside 30-day refund window"}
    return {"refunded": True, "order_id": order_id, "amount": amount}

_DISPATCH = {"lookup_order": lookup_order, "issue_refund": issue_refund}

TOOLS = [
    {"name": "lookup_order", "description": "Look up an order by its id.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"]}},
    {"name": "issue_refund", "description": "Issue a refund for an order.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"},
                      "amount": {"type": "number"}}, "required": ["order_id", "amount"]}},
]

SYSTEM = ("You are a customer-support agent. Refunds are only allowed within the "
          "30-day window. Always look up the order before issuing a refund. If a "
          "refund is not allowed, explain why politely.")

In [ ]:
from weave.conversation import Message, ToolCallPart, Usage

# --- Convert Anthropic message blocks -> weave.Message (for llm.record) ---
def _assistant_out(content):
    text = "".join(b.text for b in content if b.type == "text")
    tcs = [ToolCallPart(id=b.id, name=b.name, arguments=json.dumps(b.input))
           for b in content if b.type == "tool_use"]
    return Message.assistant(text, tool_calls=tcs or None)

def _to_weave_inputs(messages, system):
    out = [Message.system(system)]
    for m in messages:
        role, content = m["role"], m["content"]
        if role == "user" and isinstance(content, str):
            out.append(Message.user(content))
        elif role == "user":
            out += [Message.tool_result(b["tool_use_id"], b["content"]) for b in content]
        elif role == "assistant":
            out.append(Message.assistant(content) if isinstance(content, str) else _assistant_out(content))
    return out

def run_agent_turn(conversation, user_message, model=AGENT_MODEL, history=None, system=SYSTEM):
    """Run one user turn to completion.

    Returns (final_reply, transcript): the agent's text reply, plus a plain-text
    transcript of the whole trajectory (prior turns, tool calls, tool results,
    final reply) for the LLM judge to read. Pass `system` to swap the agent's
    system prompt (used to compare v1 vs v2).
    """
    messages = list(history or [])
    messages.append({"role": "user", "content": user_message})

    # Readable trajectory for the judge: prior turns first, then this turn.
    lines = [f"{m['role'].capitalize()}: {m['content']}" for m in (history or [])]
    lines.append(f"User: {user_message}")

    with conversation.start_turn(user_message=user_message) as turn:
        while True:
            with turn.start_llm(model=model, provider_name="anthropic") as llm:
                resp = _client.messages.create(
                    model=model, max_tokens=2048, system=system, tools=TOOLS, messages=messages,
                )
                llm.record(
                    input_messages=_to_weave_inputs(messages, system),
                    output_messages=[_assistant_out(resp.content)],
                    usage=Usage(input_tokens=resp.usage.input_tokens,
                                output_tokens=resp.usage.output_tokens),
                    response_id=resp.id, response_model=resp.model,
                    finish_reasons=[resp.stop_reason],
                )
            messages.append({"role": "assistant", "content": resp.content})
            for b in resp.content:
                if b.type == "text" and b.text:
                    lines.append(f"Agent: {b.text}")
                elif b.type == "tool_use":
                    lines.append(f"Agent called {b.name}({json.dumps(b.input)})")

            if resp.stop_reason != "tool_use":
                reply = "".join(b.text for b in resp.content if b.type == "text")
                return reply, "\n".join(lines)

            tool_results = []
            for block in resp.content:
                if block.type != "tool_use":
                    continue
                with turn.start_tool(name=block.name, arguments=json.dumps(block.input),
                                     tool_call_id=block.id) as tool:
                    tool.result = _DISPATCH[block.name](**block.input)
                lines.append(f"Tool result: {json.dumps(tool.result)}")
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                     "content": json.dumps(tool.result)})
            messages.append({"role": "user", "content": tool_results})

In [ ]:
import uuid
from weave.conversation import start_conversation

with start_conversation(agent_name="support-agent", conversation_id=uuid.uuid4().hex) as conv:
    reply, _ = run_agent_turn(conv, "I'd like a refund for order A1001, please.")
print(reply)

When you ran `weave.init()` above, Weave printed a **View Weave data at ...**
link. Open it and select the **Agents** tab: this conversation appears as a turn
with the model call and tool calls nested inside it.

## 2. Score the agent with an LLM judge

The score in this example is **task completion**: did the agent achieve the goal?
A judge model reads the transcript and decides against the task's success
criteria — rewarding the correct OUTCOME, not a polite-sounding reply. It's one
example score; you choose what to measure and what to name it.

In [ ]:
JUDGE_MODEL = "claude-opus-4-8"  # A different, stronger model than the agent.

def _extract_json(text):
    # Judge models sometimes wrap JSON in ``` fences; pull out the object.
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        text = text[4:] if text.startswith("json") else text
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])

def judge_task_completion(task, transcript):
    """LLM judge over the transcript. Returns {'passed': bool, 'reason': str}."""
    prompt = (
        "You are evaluating whether a customer-support agent completed the "
        "user's task. Judge the whole transcript against the success criteria "
        "— reward the correct OUTCOME, not just a polite reply.\n\n"
        f"USER REQUEST: {task['user_request']}\n"
        f"SUCCESS CRITERIA: {task['success_criteria']}\n\n"
        f"AGENT TRANSCRIPT:\n{transcript}\n\n"
        'Respond with ONLY a JSON object: {"passed": <true|false>, '
        '"reason": "<one sentence>"}. Set passed=true only if the success '
        "criteria are met."
    )
    resp = _client.messages.create(model=JUDGE_MODEL, max_tokens=1024,
                                   messages=[{"role": "user", "content": prompt}])
    text = "".join(b.text for b in resp.content if b.type == "text")
    return _extract_json(text)

Now drive the evaluation with `EvaluationLogger`. The key move: run the agent
**inside** `log_prediction(...)`, so each run's agent transcript links to the
evaluation result.

In [ ]:
TASKS = [
    {"task_id": "refund-eligible",
     "user_request": "I'd like a refund for order A1001, please.",
     "success_criteria": "Agent looks up the order and issues the refund (within the 30-day window)."},
    {"task_id": "refund-too-late",
     "user_request": "Please refund my order A1002.",
     "success_criteria": "Agent looks up the order and politely declines (outside the 30-day window); must NOT issue a refund."},
    {"task_id": "unknown-order",
     "user_request": "I want a refund for order Z9999.",
     "success_criteria": "Agent reports the order cannot be found and does not issue a refund."},
]

ev = weave.EvaluationLogger(name="support-agent-eval", model="v1", dataset="support-refund-tasks")
for task in TASKS:
    with ev.log_prediction(inputs=task) as pred:
        with start_conversation(agent_name="support-agent", conversation_id=uuid.uuid4().hex) as conv:
            reply, transcript = run_agent_turn(conv, task["user_request"])
        verdict = judge_task_completion(task, transcript)
        pred.output = reply
        pred.log_score("task_completion", verdict)
        print(f"[{task['task_id']}] passed={verdict['passed']}: {verdict['reason']}")
ev.log_summary()
print("Evaluation:", ev.ui_url)

Open the printed evaluation link, then select the **Evals** tab. Open your run's
row to reveal its details panel, which has multiple tabs:

- **Evaluation** tab — the run's scores and metrics. Click **View spans** to jump to
  the **Agents** page showing the traced spans linked to this evaluation.
- **Call** tab — a per-task breakdown with a `passed` column showing the judge's
  verdict for each task.

## 3. Organize and compare evaluations

You improve an agent by changing its **application** — the system prompt, the
tools, the control flow, or the underlying LLM — and checking whether the change
helped. To compare two versions, change the agent, then log a second evaluation
under the same `name` with a new version label.

`model` is Weave's term for the system under evaluation — your whole agent
(prompt, tools, code, and LLM), not just the LLM. Reusing the eval `name` and
changing the label aligns the runs as versions. (To record *what* changed, you
can pass a dict, e.g. `model={"name": "support-agent", "version": "v2",
"system_prompt": SYSTEM_V2}`, and Weave stores those as versioned attributes.)

Below, we re-run the same tasks and judge with a revised system prompt, logged
under the label `v2`:

In [ ]:
# v2: a stricter system prompt. Same tasks, loop, and judge as v1.
SYSTEM_V2 = SYSTEM + (" Keep replies to one or two sentences and state the refund "
                      "decision explicitly.")

ev = weave.EvaluationLogger(name="support-agent-eval", model="v2", dataset="support-refund-tasks")
for task in TASKS:
    with ev.log_prediction(inputs=task) as pred:
        with start_conversation(agent_name="support-agent", conversation_id=uuid.uuid4().hex) as conv:
            reply, transcript = run_agent_turn(conv, task["user_request"], system=SYSTEM_V2)
        verdict = judge_task_completion(task, transcript)
        pred.output = reply
        pred.log_score("task_completion", verdict)
        print(f"[{task['task_id']}] passed={verdict['passed']}: {verdict['reason']}")
ev.log_summary()
print("Evaluation:", ev.ui_url)

## 4. Score a multi-turn conversation

Real conversations span several turns, and a capable agent carries context
forward — it shouldn't re-ask for an order id the user already gave. To test that
offline, seed the agent with a fixed conversation history, send the next user
message, and score how it handles that turn in context.

Each dataset row is one such scenario: the prior turns plus the next message the
agent must answer. Below, the order id appears only in the history, so a good
agent reuses it instead of asking again.

In [ ]:
MULTI_TURN_TASKS = [
    {"task_id": "context-refund",
     "conversation_history": [
         {"role": "user", "content": "Hi, can you check the status of my order A1001?"},
         {"role": "assistant", "content": "Your order A1001 (Wireless headphones) was delivered 5 days ago."},
     ],
     "next_user_message": "Thanks. Actually, I'd like to return it for a refund.",
     "success_criteria": "Using the prior context that the order is A1001, the agent issues the refund (within 30 days) WITHOUT asking the user to repeat the order id."},
    {"task_id": "context-decline",
     "conversation_history": [
         {"role": "user", "content": "Can you look up order A1002 for me?"},
         {"role": "assistant", "content": "Order A1002 (Standing desk) was delivered 60 days ago."},
     ],
     "next_user_message": "Okay, I'd like a refund for it.",
     "success_criteria": "Using the prior context that A1002 was delivered 60 days ago, the agent politely declines as outside the 30-day window and does NOT issue a refund."},
]

ev = weave.EvaluationLogger(name="support-agent-multiturn-eval", model="v1", dataset="support-multiturn-tasks")
for task in MULTI_TURN_TASKS:
    with ev.log_prediction(inputs=task) as pred:
        with start_conversation(agent_name="support-agent", conversation_id=uuid.uuid4().hex) as conv:
            reply, transcript = run_agent_turn(conv, task["next_user_message"],
                                               history=task["conversation_history"])
        judge_task = {"user_request": task["next_user_message"], "success_criteria": task["success_criteria"]}
        verdict = judge_task_completion(judge_task, transcript)
        pred.output = reply
        pred.log_score("task_completion", verdict)
        print(f"[{task['task_id']}] passed={verdict['passed']}: {verdict['reason']}")
ev.log_summary()
print("Evaluation:", ev.ui_url)

Open the printed link, then select the **Evals** tab. Open your run's *-multiturn-eval'
row to reveal its details panel.  Select the **Evaluation** tab and explore the two examples showing the conversation and the passing scorer.

## Recap

You traced an agent as a conversation, scored task completion for single- and
multi-turn interactions, and compared two versions — all linked back to the agent
transcripts.